In [34]:
%reload_ext autoreload
%autoreload 2

In [47]:
import sys
import numpy as np

path_pjt = (
    "/Users/Nguye071/Documents/GitHub/Single-and-Multi-level-Fourier-RQMC-for-MSRM"
)
sys.path.insert(0, path_pjt)
sys.path.append(path_pjt + "/3D NIG QPC Loss")
from QMC_NIG_QPC import RQMC_Fou_NIG_qpc
from QMC_CV_NIG_QPC import RQMC_CV_Fou_NIG_qpc
from MC_loss import MCLossFunction
from generate_NIG_rv import MNIGSampler
from scipy.optimize import minimize

# 3D case

In [ ]:
maxiter = 3500
mu = [0.00084, 0.00024, 0.00055]  # location vector 3x1
sigma = [
    [2.338, 1.796, 2.08],
    [1.796, 2.327, 2.088],
    [2.08, 2.088, 2.555],
]  ## 3x3 covariance matrix
alpha = 1.0  #
alpha1 = 365.78  # Tail parameter
beta = [-64.27, 41.45, 7.35]  # skewness vector
delta = 0.373  # scale (mixing) parameter
c = 1.0  # Optional constant for the inequality constraint
dim = len(mu)
guess = np.zeros(dim)  ## Initial value for optimization procedure
epsilon = 10  # Scale used in domain transformation
N_sobol = 1024  # number of Sobol points
m_shift = 32  # number of randomized digital shifts

In [ ]:
# =========================
# Print helpers for damping
# =========================
def print_1d_history(vals, title, show=6, ind="damping"):
    print(title)
    print("-" * len(title))
    n = len(vals)
    head = min(show, n)
    for it, d in enumerate(vals[:head]):
        print(f"iter {it:3d} | {ind} = {d:10.4g}")
    if n > show:
        print(f"... ({n - show} more iterations not shown)")


def print_2d_history(vals, title, show=6):
    print(title)
    print("-" * len(title))
    n = len(vals)
    head = min(show, n)
    for it, tup in enumerate(vals[:head]):
        tup_str = "(" + ", ".join(f"{x:10.4g}" for x in tup) + ")"
        print(f"iter {it:3d} | tuple = {tup_str}")
    if n > show:
        print(f"... ({n - show} more iterations not shown)")

## Single-level Fourier-RQMC

In [ ]:
# =========================
# Build single-level loss object, here for numerical convenience we fix the damping at the initial level
# =========================
loss_qmc_Fou_qpc_3D = RQMC_Fou_NIG_qpc(
    v_mu=mu,
    v_sigma=sigma,
    v_alpha=alpha1,
    v_beta=beta,
    v_delta=delta,
    N_sobol=N_sobol,
    m_shift=m_shift,
    alpha=alpha,
    c=c,
    epsilon=epsilon,
)


# =========================
# Internal bookkeeping helpers at each iteration
# =========================
def _set_current_m(m):
    """Store current iterate information"""
    m = np.asarray(m, dtype=float)
    loss_qmc_Fou_qpc_3D._current_m = m

    if loss_qmc_Fou_qpc_3D._pending_jac is not None:
        loss_qmc_Fou_qpc_3D._jac_components.append(loss_qmc_Fou_qpc_3D._pending_jac)
        loss_qmc_Fou_qpc_3D._pending_jac = None

    loss_qmc_Fou_qpc_3D.record_qmc_stats()


## Objective: \sum_{k=1}^{d} m_k
def objective_wrapper(m):
    m = np.asarray(m, dtype=float)
    loss_qmc_Fou_qpc_3D._current_m = m.copy()
    return loss_qmc_Fou_qpc_3D.objective(m)


## Gradient of the objective
def jac_wrapper(m):
    m = np.asarray(m, dtype=float)
    loss_qmc_Fou_qpc_3D._current_m = m.copy()
    return loss_qmc_Fou_qpc_3D.objective_jac(m)


def scipy_callback(m):
    _set_current_m(m)


# =========================
# Warm-up evaluation at initial point of the optimization
# =========================
loss_qmc_Fou_qpc_3D._current_m = np.asarray(guess, dtype=float)
loss_qmc_Fou_qpc_3D.shortfall_risk(guess)
loss_qmc_Fou_qpc_3D.shortfall_risk_jac(guess)
_set_current_m(guess)

# =========================
# Constraint E[l(X - m)] and its gradient E[∇l(X - m)]
# =========================
cons_qmc_exact = {
    "type": "ineq",
    "fun": lambda x: loss_qmc_Fou_qpc_3D.ineq_constraint(x),
    "jac": lambda x: loss_qmc_Fou_qpc_3D.ineq_constraint_jac(x),
}

# =========================
# Optimization
# =========================
res = minimize(
    fun=objective_wrapper,
    x0=guess,
    jac=jac_wrapper,
    constraints=cons_qmc_exact,
    method="SLSQP",
    options={"maxiter": maxiter, "ftol": 1e-6},
    callback=scipy_callback,
)


# =========================
# Damping histories
# =========================
print(f"Number of iterations: {res.nit}")

## Loss 1D components and its gradient
vals_1d_g = loss_qmc_Fou_qpc_3D._history_1d["g"][0]["K"]
vals_1d_f = loss_qmc_Fou_qpc_3D._history_1d["f"][0]["K"]

print_1d_history(
    vals_1d_g,
    "First 1D loss Fourier component damping by iteration",
    show=6,
)

print_1d_history(
    vals_1d_f,
    "First 1D loss gradient Fourier component damping by iteration",
    show=3,
)

## Loss 2D components and its gradient

vals_2d_h = loss_qmc_Fou_qpc_3D._history_2d["h"][(0, 1)]["K"]
vals_2d_l = loss_qmc_Fou_qpc_3D._history_2d["l"][(0, 1)]["K"]
print_2d_history(
    vals_2d_h,
    "First 2D loss Fourier component damping K by iteration",
    show=3,
)
print_2d_history(
    vals_2d_l,
    "First 2D loss gradient Fourier component damping K by iteration",
    show=3,
)
# =========================
# Final solution + relative error
# =========================
max_qmc_err = loss_qmc_Fou_qpc_3D.statistical_error_sol_RQMC(res.x)
rel_err = max_qmc_err / np.abs(np.max(res.x))

print("\nFinal result")
print("------------")
print(f"Single Fourier-RQMC solution: {res.x}")
print(f"Relative statistical error : {rel_err:.6g}")

/Users/Nguye071/.global_python_env/lib/python3.13/site-packages/qmcpy/discrete_distribution/digital_net_b2/digital_net_b2.py:389


Number of iterations: 25
First 1D loss Fourier component damping by iteration
----------------------------------------------------
iter   0 | damping =      3.664
iter   1 | damping =      3.664
iter   2 | damping =      3.664
iter   3 | damping =      3.664
iter   4 | damping =      3.664
iter   5 | damping =      3.664
... (20 more iterations not shown)
First 1D loss gradient Fourier component damping by iteration
-------------------------------------------------------------
iter   0 | damping =       2.68
iter   1 | damping =       2.68
iter   2 | damping =       2.68
... (23 more iterations not shown)
First 2D loss Fourier component damping K by iteration
------------------------------------------------------
iter   0 | tuple = (     2.135,      2.139)
iter   1 | tuple = (      1.82,      1.824)
iter   2 | tuple = (     1.859,      1.862)
... (23 more iterations not shown)
First 2D loss gradient Fourier component damping K by iteration
----------------------------------------------

## Multilevel Fourier-RQMC

In [ ]:
# =========================
# Build multilevel Fourier-RQMC loss object
# =========================
loss_qmc_CV_Fou_qpc_3D = RQMC_CV_Fou_NIG_qpc(
    v_mu=mu,
    v_sigma=sigma,
    v_alpha=alpha1,
    v_beta=beta,
    v_delta=delta,
    N_sobol=N_sobol,
    m_shift=m_shift,
    alpha=alpha,
    c=c,
    epsilon=4,
)

# =========================
# Internal bookkeeping helpers at each iteration
# =========================
#
N_sobol_iter = []  # List to keep track for number of Sobol points
count = 0  # This will use to keep track when we enter the local convergence region


def _set_current_m(m):
    """Store current iterate."""
    global count
    m = np.asarray(m, dtype=float)
    loss_qmc_CV_Fou_qpc_3D._current_m = m

    # The commit option in this case help us to store only succesful optimization iteration
    loss_qmc_CV_Fou_qpc_3D.shortfall_risk(m, commit=True)
    loss_qmc_CV_Fou_qpc_3D.shortfall_risk_jac(m, commit=True)
    if loss_qmc_CV_Fou_qpc_3D._pending_jac is not None:
        loss_qmc_CV_Fou_qpc_3D._jac_components.append(
            loss_qmc_CV_Fou_qpc_3D._pending_jac
        )
    loss_qmc_CV_Fou_qpc_3D._pending_jac = None
    count += 1

    ## Adaptive choice of Sobol point when entering local convergence region
    N_sobol_iter.append(loss_qmc_CV_Fou_qpc_3D._N_current_loss)
    if count >= 8:
        loss_qmc_CV_Fou_qpc_3D._divide_sobol(mult_factor=1 / 8)
        loss_qmc_CV_Fou_qpc_3D._divide_sobol(mult_factor=1 / 8, grad=True)

    loss_qmc_CV_Fou_qpc_3D.record_qmc_stats()


## Objective: \sum_{k=1}^{d} m_k
def objective_wrapper(m):
    m = np.asarray(m, dtype=float)
    loss_qmc_CV_Fou_qpc_3D._current_m = m.copy()
    return loss_qmc_CV_Fou_qpc_3D.objective(m)


## Gradient of the objective
def jac_wrapper(m):
    m = np.asarray(m, dtype=float)
    loss_qmc_CV_Fou_qpc_3D._current_m = m.copy()
    return loss_qmc_CV_Fou_qpc_3D.objective_jac(m)


def scipy_callback(m):
    _set_current_m(m)


# =========================
# Warm-up evaluation at initial point of the optimization
# =========================
loss_qmc_CV_Fou_qpc_3D._current_m = np.asarray(guess, dtype=float)
loss_qmc_CV_Fou_qpc_3D.shortfall_risk(guess)
loss_qmc_CV_Fou_qpc_3D.shortfall_risk_jac(guess)
_set_current_m(guess)

# =========================
# Constraint E[l(X - m)] and its gradient E[∇l(X - m)]
# =========================
cons_qmc_exact = {
    "type": "ineq",
    "fun": lambda x: loss_qmc_CV_Fou_qpc_3D.ineq_constraint(x),
    "jac": lambda x: loss_qmc_CV_Fou_qpc_3D.ineq_constraint_jac(x),
}

# =========================
# Optimization
# =========================
res = minimize(
    fun=objective_wrapper,
    x0=guess,
    jac=jac_wrapper,
    constraints=cons_qmc_exact,
    method="SLSQP",
    options={"maxiter": maxiter, "ftol": 1e-6},
    callback=scipy_callback,
)


# =========================
# Damping and Sobol point histories
# =========================
print(f"Number of iterations: {res.nit}")
## Loss 1D components and its gradient
vals_1d_g = loss_qmc_CV_Fou_qpc_3D._history_1d["g"][0]["K"]
vals_1d_f = loss_qmc_CV_Fou_qpc_3D._history_1d["f"][0]["K"]

print_1d_history(
    vals_1d_g,
    "First 1D loss Fourier component damping by iteration",
    show=3,
)

print_1d_history(
    vals_1d_f,
    "First 1D loss gradient Fourier component damping by iteration",
    show=3,
)

## Loss 2D components and its gradient

vals_2d_h = loss_qmc_CV_Fou_qpc_3D._history_2d["h"][(0, 1)]["K"]
vals_2d_l = loss_qmc_CV_Fou_qpc_3D._history_2d["l"][(0, 1)]["K"]
print_2d_history(
    vals_2d_h,
    "First 2D loss Fourier component damping K by iteration",
    show=3,
)

Number of iterations: 14
First 1D loss Fourier component damping by iteration
----------------------------------------------------
iter   0 | damping =      3.422
iter   1 | damping =      3.422
iter   2 | damping =      3.422
... (12 more iterations not shown)
First 1D loss gradient Fourier component damping by iteration
-------------------------------------------------------------
iter   0 | damping =      2.422
iter   1 | damping =      2.422
iter   2 | damping =      2.422
... (12 more iterations not shown)
First 2D loss Fourier component damping K by iteration
------------------------------------------------------
iter   0 | tuple = (     1.526,      1.529)
iter   1 | tuple = (     1.353,      1.358)
iter   2 | tuple = (     1.559,      1.565)
... (12 more iterations not shown)


In [ ]:
print_2d_history(
    vals_2d_l,
    "First 2D loss gradient Fourier component damping K by iteration",
    show=3,
)

## Sobol points
print_1d_history(N_sobol_iter, "Number of Sobol points", show=13, ind="N_Sobol")
# =========================
# Final solution + relative error
# =========================
max_qmc_err = loss_qmc_CV_Fou_qpc_3D.statistical_error_sol_RQMC(res.x, loc_index=8)
rel_err = max_qmc_err / np.abs(np.max(res.x))

print("\nFinal result")
print("------------")
print(f"Multilevel Fourier-RQMC solution: {res.x}")
print(f"Relative statistical error : {rel_err:.6g}")

First 2D loss gradient Fourier component damping K by iteration
---------------------------------------------------------------
iter   0 | tuple = (     1.693,     0.9232)
iter   1 | tuple = (     1.488,     0.7907)
iter   2 | tuple = (     1.733,     0.9496)
... (12 more iterations not shown)
Number of Sobol points
----------------------
iter   0 | N_Sobol =       1024
iter   1 | N_Sobol =       1024
iter   2 | N_Sobol =       1024
iter   3 | N_Sobol =       1024
iter   4 | N_Sobol =       1024
iter   5 | N_Sobol =       1024
iter   6 | N_Sobol =       1024
iter   7 | N_Sobol =       1024
iter   8 | N_Sobol =        128
iter   9 | N_Sobol =         16
iter  10 | N_Sobol =          4
iter  11 | N_Sobol =          4
iter  12 | N_Sobol =          4
... (2 more iterations not shown)

Final result
------------
Multilevel Fourier-RQMC solution: [-0.25289356 -0.25160219 -0.24853998]
Relative statistical error : 1.7321


## SAA

In [ ]:
import math
import scipy.stats

MC_points = 10**6  ## Number of MC points
rv = MNIGSampler(mu=mu, Sigma=sigma, alpha=alpha1, beta=beta, delta=delta)

X = rv.sample(MC_points)


# =========================
# Build SAA loss object
# =========================
loss_MC_3D = MCLossFunction(X, alpha=alpha, c=c)


def _set_current_m(m):
    """Store current iterate."""
    m = np.asarray(m, dtype=float)
    loss_MC_3D._current_m = m
    loss_MC_3D._var_iter.append(loss_MC_3D._pending_var)
    loss_MC_3D._pending_var = None


# =========================
# Internal bookkeeping helpers at each iteration
# =========================

## Objective: \sum_{k=1}^{d} m_k


def objective_wrapper(m):
    m = np.asarray(m, dtype=float)
    loss_MC_3D._current_m = m.copy()
    return loss_MC_3D.objective(m)


## Gradient of the objective
def jac_wrapper(m):
    m = np.asarray(m, dtype=float)
    loss_MC_3D._current_m = m.copy()
    return loss_MC_3D.objective_jac(m)


def scipy_callback(m):
    _set_current_m(m)


# =========================
# Warm-up evaluation at initial point of the optimization
# =========================
loss_MC_3D._current_m = guess
loss_MC_3D.shortfall_risk(guess)
loss_MC_3D.shortfall_risk_jac(guess)
loss_MC_3D._var_iter.append(loss_MC_3D._pending_var)

# =========================
# Constraint E[l(X - m)] and its gradient E[∇l(X - m)]
# =========================
cons = {
    "type": "ineq",
    "fun": lambda x: loss_MC_3D.ineq_constraint(x),
    "jac": lambda x: loss_MC_3D.ineq_constraint_jac(x),
}

# =========================
# Optimization
# =========================
res = minimize(
    fun=objective_wrapper,
    x0=guess,
    jac=jac_wrapper,
    constraints=cons,
    method="SLSQP",
    options={"maxiter": maxiter, "ftol": 1e-6},
    callback=scipy_callback,
)

# =========================
# Final solution + relative error
# =========================
print(f"Number of iterations: {res.nit}")
max_mc_err = loss_MC_3D.statistical_error_sol_MC(res.x)
rel_err = max_mc_err / np.abs(np.max(res.x))

print("\nFinal result")
print("------------")

print(f"SAA solution: {res.x}")
print(f"Relative statistical error : {rel_err:.6g}")

Number of iterations: 5

Final result
------------
SAA solution: [-0.2416766  -0.24177864 -0.24115328]
Relative statistical error : 105.169
